In [229]:
import pymc as pm
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import joblib

import arviz as az
import matplotlib.pyplot as plt


## Data 

In [241]:
data = pd.read_csv("../../inputs/rolled_player_data_23-24_all_players_xp_fixed.csv")
data = data[(data["round"] > 3) & (data["minutes_rolling_5"] > 0)]

# Columns kept per position. Must include the numeric features listed in src/model_dispatcher.py.
GK_FEATS = [
    # "was_home",
    "round",
    "elo_h",
    "elo_a",
    "team_h_difficulty",
    "team_a_difficulty",
    "win_prob",
    "value",
    "selected_by_percent",
    "transfers_in",
    "transfers_out",
    "xP_rolling_3",
    "xGChain_rolling_3",
    "xGBuildup_rolling_3",
    "minutes_rolling_3",
    "starts_rolling_3",
    "expected_goals_conceded_rolling_3",
    "clean_sheets_rolling_3",
    "saves_rolling_3",
    "ict_index_rolling_3",
    "influence_rolling_3",
    "creativity_rolling_3",
    "threat_rolling_3",
]
DEF_FEATS = [
    # "was_home",
    "round",
    "elo_h",
    "elo_a",
    "team_h_difficulty",
    "team_a_difficulty",
    "win_prob",
    "value",
    "selected_by_percent",
    "transfers_in",
    "transfers_out",
    "xP_rolling_3",
    "xGChain_rolling_3",
    "xGBuildup_rolling_3",
    "expected_goals_rolling_3",
    "expected_assists_rolling_3",
    "expected_goal_involvements_rolling_3",
    "minutes_rolling_3",
    "starts_rolling_3",
    "expected_goals_conceded_rolling_3",
    "clean_sheets_rolling_3",
    "saves_rolling_3",
    "ict_index_rolling_3",
    "influence_rolling_3",
    "creativity_rolling_3",
    "threat_rolling_3",
]
MID_FEATS = [
    # "was_home",
    "round",
    "elo_h",
    "elo_a",
    "team_h_difficulty",
    "team_a_difficulty",
    "win_prob",
    "value",
    "selected_by_percent",
    "transfers_in",
    "transfers_out",
    "xP_rolling_3",
    "xGChain_rolling_3",
    "xGBuildup_rolling_3",
    "expected_goals_rolling_3",
    "expected_assists_rolling_3",
    "expected_goal_involvements_rolling_3",
    "minutes_rolling_3",
    "starts_rolling_3",
    "ict_index_rolling_3",
    "influence_rolling_3",
    "threat_rolling_3",
    "creativity_rolling_3",
]
FWD_FEATS = [
    # "was_home",
    "round",
    "elo_h",
    "elo_a",
    "team_h_difficulty",
    "team_a_difficulty",
    "win_prob",
    "value",
    "selected_by_percent",
    "transfers_in",
    "transfers_out",
    "xP_rolling_3",
    "xGChain_rolling_3",
    "xGBuildup_rolling_3",
    "expected_goals_rolling_3",
    "expected_assists_rolling_3",
    "expected_goal_involvements_rolling_3",
    "minutes_rolling_3",
    "starts_rolling_3",
    "ict_index_rolling_3",
    "influence_rolling_3",
    "threat_rolling_3",
    "creativity_rolling_3",
]

gk_data = data[data["position"] == "GK"][GK_FEATS]
def_data = data[data["position"] == "DEF"][DEF_FEATS]
mid_data = data[data["position"] == "MID"][MID_FEATS]
fwd_data = data[data["position"] == "FWD"][FWD_FEATS]


gk_data["select_prob"] = gk_data["selected_by_percent"] / gk_data.groupby("round")[
    "selected_by_percent"
].transform("sum")

def_data["select_prob"] = def_data["selected_by_percent"] / def_data.groupby("round")[
    "selected_by_percent"
].transform("sum")

mid_data["select_prob"] = mid_data["selected_by_percent"] / mid_data.groupby("round")[
    "selected_by_percent"
].transform("sum")

fwd_data["select_prob"] = fwd_data["selected_by_percent"] / fwd_data.groupby("round")[
    "selected_by_percent"
].transform("sum")


epsilon = 1e-9  # A very small number
gk_data.loc[gk_data["select_prob"] == 0, "select_prob"] = epsilon
def_data.loc[def_data["select_prob"] == 0, "select_prob"] = epsilon
mid_data.loc[mid_data["select_prob"] == 0, "select_prob"] = epsilon
fwd_data.loc[fwd_data["select_prob"] == 0, "select_prob"] = epsilon


# gk_data = gk_data.drop(columns=["round"])
# def_data = def_data.drop(columns=["round"])
# mid_data = mid_data.drop(columns=["round"])
# fwd_data = fwd_data.drop(columns=["round"])


gk_observed_p = gk_data["select_prob"]
def_observed_p = def_data["select_prob"]
mid_observed_p = mid_data["select_prob"]
fwd_observed_p = fwd_data["select_prob"]


### Scale

In [231]:
gk_data

,was_home,round,elo_h,elo_a,team_h_difficulty,team_a_difficulty,win_prob,value,selected_by_percent,transfers_in,...,minutes_rolling_3,starts_rolling_3,expected_goals_conceded_rolling_3,clean_sheets_rolling_3,saves_rolling_3,ict_index_rolling_3,influence_rolling_3,creativity_rolling_3,threat_rolling_3,select_prob
597,True,6,1681.25,1583.69,3,5,0.565,50,14.3,12952,...,60.0,0.666667,0.496667,0.000000,0.666667,0.766667,7.666667,0.0,0.0,0.078485
598,False,7,1334.90,1676.89,5,2,0.618,49,12.2,7214,...,30.0,0.333333,0.320000,0.000000,0.333333,0.366667,3.600000,0.0,0.0,0.066813
599,True,8,1683.21,1809.81,5,5,0.324,49,10.6,3542,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.057955
600,False,9,1489.09,1694.97,5,3,0.423,48,9.7,3625,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.052977
605,True,14,1693.07,1428.20,2,5,0.751,47,7.5,12249,...,30.0,0.333333,0.370000,0.333333,0.333333,0.166667,1.600000,0.0,0.0,0.040939
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28213,False,34,1411.90,1384.69,2,2,0.264,45,0.1,2339,...,90.0,1.000000,1.343333,0.000000,2.666667,2.033333,20.400000,0.0,0.0,0.000648
28214,True,35,1373.58,1813.56,5,2,0.118,45,0.1,624,...,90.0,1.000000,1.000000,0.000000,2.666667,2.033333,20.466667,0.0,0.0,0.000610
28215,False,36,1259.09,1370.48,2,2,0.575,45,0.2,5726,...,90.0,1.000000,0.786667,0.000000,2.000000,1.600000,16.200000,0.0,0.0,0.001212
28216,True,37,1383.49,1547.40,3,2,0.278,45,0.1,1035,...,90.0,1.000000,1.186667,0.000000,3.000000,2.300000,23.000000,0.0,0.0,0.000604


In [237]:
# 1. Standardize (PCA works best on standardized data)
scaler_gk = StandardScaler(with_mean=True, with_std=True)
scaler_def = StandardScaler(with_mean=True, with_std=True)
scaler_mid = StandardScaler(with_mean=True, with_std=True)
scaler_fwd = StandardScaler(with_mean=True, with_std=True)

# print(gk_data.shape)
X_gk_rounds = gk_data[["round"]]
X_gk_data = gk_data.copy()
X_gk_data = X_gk_data.drop(columns=["round", "was_home"])
X_gk = pd.concat(
    [
        X_gk_rounds.reset_index(drop=True),
        pd.DataFrame(
            scaler_gk.fit_transform(X_gk_data.values), columns=X_gk_data.columns
        ).reset_index(drop=True),
    ],
    axis=1,
)


X_def = pd.DataFrame(
    scaler_def.fit_transform(def_data.values), columns=def_data.columns
)


X_mid = pd.DataFrame(
    scaler_mid.fit_transform(mid_data.values), columns=mid_data.columns
)
X_fwd = pd.DataFrame(
    scaler_fwd.fit_transform(fwd_data.values), columns=fwd_data.columns
)

# save standardscaler model
joblib.dump(scaler_gk, "scaler_gk.pkl")
joblib.dump(scaler_def, "scaler_def.pkl")
joblib.dump(scaler_mid, "scaler_mid.pkl")
joblib.dump(scaler_fwd, "scaler_fwd.pkl")

['scaler_fwd.pkl']

In [238]:
X_gk  # .corrwith(X_gk["select_prob"]).abs()

,round,elo_h,elo_a,team_h_difficulty,team_a_difficulty,win_prob,value,selected_by_percent,transfers_in,transfers_out,...,minutes_rolling_3,starts_rolling_3,expected_goals_conceded_rolling_3,clean_sheets_rolling_3,saves_rolling_3,ict_index_rolling_3,influence_rolling_3,creativity_rolling_3,threat_rolling_3,select_prob
0,6,1.306871,0.599661,0.352978,1.728746,0.849892,0.869497,0.997040,-0.385592,4.118352,...,-0.231168,-0.225561,-0.905433,-0.685693,-1.047366,-0.917197,-0.920776,-0.241602,-0.034731,0.839183
1,7,-1.229510,1.295103,2.387506,-0.944905,1.109279,0.653489,0.721459,-0.512119,4.240218,...,-1.211038,-1.180743,-1.153502,-0.685693,-1.250193,-1.232903,-1.250068,-0.241602,-0.034731,0.586244
2,8,1.321224,2.286929,2.387506,1.728746,-0.329585,0.653489,0.511493,-0.593090,3.183367,...,-2.190907,-2.135925,-1.602835,-0.685693,-1.453019,-1.522301,-1.541573,-0.241602,-0.034731,0.394304
3,9,-0.100350,1.430013,2.387506,-0.053688,0.154931,0.437481,0.393387,-0.591260,1.896628,...,-2.190907,-2.135925,-1.602835,-0.685693,-1.453019,-1.522301,-1.541573,-0.241602,-0.034731,0.286419
4,14,1.393431,-0.560577,-0.664286,1.728746,1.760194,0.221473,0.104683,-0.401094,-0.066270,...,-1.211038,-1.180743,-1.083294,0.812005,-1.250193,-1.390757,-1.412015,-0.241602,-0.034731,0.025567
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
825,34,-0.665626,-0.885241,-0.664286,-0.944905,-0.623231,-0.210543,-0.866412,-0.619617,-0.642928,...,0.748702,0.729621,0.283427,-0.685693,0.169592,0.082540,0.110287,-0.241602,-0.034731,-0.847522
826,35,-0.946250,2.314910,2.387506,-0.944905,-1.337769,-0.210543,-0.866412,-0.657434,-0.616350,...,0.748702,0.729621,-0.198670,-0.685693,0.169592,0.082540,0.115685,-0.241602,-0.034731,-0.848353
827,36,-1.784680,-0.991274,-0.664286,-0.944905,0.898833,-0.210543,-0.853289,-0.544931,-0.649934,...,0.748702,0.729621,-0.498225,-0.685693,-0.236061,-0.259475,-0.229802,-0.241602,-0.034731,-0.835299
828,37,-0.873677,0.328872,0.352978,-0.944905,-0.554713,-0.210543,-0.866412,-0.648371,-0.569381,...,0.748702,0.729621,0.063441,-0.685693,0.372419,0.293011,0.320818,-0.241602,-0.034731,-0.848472


### PCA

In [234]:
# 2. Choose # of components or variance threshold
pca_gk = PCA(n_components=0.95)  # 0.95 = keep enough PCs to explain 95% of variance
pca_def = PCA(n_components=0.95)
pca_mid = PCA(n_components=0.95)
pca_fwd = PCA(n_components=0.95)

# 3. Fit PCA and transform
X_gk_pca = pd.DataFrame(pca_gk.fit_transform(X_gk))
X_def_pca = pd.DataFrame(pca_def.fit_transform(X_def))
X_mid_pca = pd.DataFrame(pca_mid.fit_transform(X_mid))
X_fwd_pca = pd.DataFrame(pca_fwd.fit_transform(X_fwd))

# save pca mode;
joblib.dump(pca_gk, "pca_gk.pkl")
joblib.dump(pca_def, "pca_def.pkl")
joblib.dump(pca_mid, "pca_mid.pkl")
joblib.dump(pca_fwd, "pca_fwd.pkl")

print("Original shape:", X_gk.shape, X_def.shape, X_mid.shape, X_fwd.shape)
print(
    "Reduced shape:", X_gk_pca.shape, X_def_pca.shape, X_mid_pca.shape, X_fwd_pca.shape
)

ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
import pytensor.tensor as pt


def select_features_by_correlation(
    df,
    candidate_feats,
    target_col="select_prob",
    redundancy_threshold=0.8,
    min_corr=0.0,
    verbose=True,
):
    """
    Automatic feature selection in two steps:
      1. Rank candidate_feats by |correlation| with target_col, descending
         -- strongest predictors of xP first.
      2. Walk down the ranked list, greedily keeping a feature only if its
         |correlation| with every already-selected feature is below
         redundancy_threshold. A feature that's individually predictive of
         xP but redundant with something already kept (i.e. it's not
         adding new information) gets dropped.

    This is a filter-based analogue of stepwise selection: it favors
    features that are both informative about the target and
    non-redundant with each other, without needing to actually fit the
    (expensive) Dirichlet model repeatedly.

    Parameters
    ----------
    df                    : DataFrame containing candidate_feats + target_col
    candidate_feats       : list of column names to consider
    target_col            : column to rank relevance against (e.g. 'xP')
    redundancy_threshold  : drop a feature if |corr| with an already-kept
                             feature exceeds this (0-1). Lower = stricter
                             (keeps fewer, more independent features).
    min_corr              : also require |corr| with target_col to exceed
                             this floor to be considered at all (0-1).
                             Default 0.0 = no relevance floor, matching a
                             pure rank-then-deduplicate selection.

    Returns
    -------
    X_feats : list of selected feature names, in original candidate order
    """

    df_valid = df[candidate_feats + [target_col]].dropna()
    if len(df_valid) < len(df):
        dropped = len(df) - len(df_valid)
        if verbose:
            print(f"  dropped {dropped} rows with NaNs before computing correlations")

    target_corr = df_valid[candidate_feats].corrwith(df_valid[target_col]).abs()

    ranked = target_corr[~target_corr.index.duplicated(keep="first")].sort_values(
        ascending=False
    )  # target_corr.sort_values(ascending=False)

    feat_corr = df_valid[candidate_feats].corr().abs()

    selected = []
    # print(ranked.index)
    for feat in ranked.index:
        if pd.isna(ranked[feat]):
            if verbose:
                print(
                    f"  skipping {feat!r} -- undefined correlation with {target_col} "
                    f"(likely constant column)"
                )
            continue
        if ranked[feat] < min_corr:
            if verbose:
                print(
                    f"  skipping {feat!r} -- |corr| with {target_col} "
                    f"({ranked[feat]:.3f}) below min_corr={min_corr}"
                )
            continue
        if not selected:
            selected.append(feat)
            continue
        max_corr_with_selected = feat_corr.loc[feat, selected].max()
        if max_corr_with_selected < redundancy_threshold:
            selected.append(feat)
        elif verbose:
            worst = feat_corr.loc[feat, selected].idxmax()
            print(
                f"  dropping {feat!r} (|corr| with {target_col}={target_corr[feat]:.3f}) "
                f"-- |corr|={max_corr_with_selected:.3f} with already-selected {worst!r}"
            )

    # restore original candidate order for readability/downstream indexing
    X_feats = [f for f in candidate_feats if f in selected]

    if verbose:
        print(
            f"\nSelected {len(X_feats)} / {len(candidate_feats)} features "
            f"(ranked by |corr| with {target_col!r}, redundancy_threshold={redundancy_threshold}, "
            f"min_corr={min_corr}):"
        )
        print(X_feats)

    return X_feats


"""
Cross-sectional league baseline: fit -> standardize -> prune -> refit pipeline,
generalized across positions (g, d, m, f). Wraps the exact recipe that worked
for goalkeepers so it can be reused for d/m/f without re-deriving it each time.
"""


def fit_dirichlet_regression(
    X,
    select_probs,
    n_features,
    sigma=1.0,
    draws=2000,
    tune=2000,
    chains=4,
    cores=4,
    target_accept=0.99,
    max_treedepth=12,
    random_seed=None,
):
    """
    One NUTS fit of p ~ Dirichlet(exp(X @ beta)), beta ~ Normal(0, sigma).
    Returns (trace, summary_df).
    """
    with pm.Model() as model:
        beta = pm.Normal(
            "beta", mu=0.0, sigma=sigma, shape=n_features
        )  # pm.Flat("beta", shape=n_features)

        alpha = pm.math.exp(pt.dot(X, beta))
        alpha = pt.clip(alpha, 1e-6, 1e6)
        pm.Dirichlet("p", a=alpha, observed=select_probs)

        trace = pm.sample(
            draws,
            tune=tune,
            chains=chains,
            cores=cores,
            target_accept=target_accept,
            max_treedepth=max_treedepth,
            random_seed=random_seed,
        )

    summary = az.summary(trace, var_names=["beta"])
    n_div = int(trace.sample_stats["diverging"].sum())
    print(f"  divergences: {n_div}")
    if n_div > 0:
        print("  WARNING: divergences present -- treat this fit as provisional")
    return trace, summary


def _hdi_overlap(lo1, hi1, lo2, hi2):
    """
    Fraction of the *narrower* interval's width that's covered by the
    overlap with the other interval. 1.0 = one interval fully contains
    (or exactly equals) the other; 0.0 = no overlap at all.
    """
    overlap = max(0.0, min(hi1, hi2) - max(lo1, lo2))
    narrower_width = min(hi1 - lo1, hi2 - lo2)
    if narrower_width <= 0:
        return 0.0
    return overlap / narrower_width


def prune_by_hdi(summary, feats, overlap_threshold=0.8, verbose=True):
    """
    Keep only features whose 94% HDI excludes zero, AND deduplicate
    features whose HDIs are near-identical to one another (a sign the
    two are capturing essentially the same effect -- often leftover
    collinearity that survived the raw-feature correlation check,
    since it can also arise from *combinations* of correlated features
    rather than a single pairwise correlation).

    Greedy strategy: process candidates in order of decreasing |mean|
    (strongest effect first). A candidate is dropped if its HDI overlaps
    an already-kept feature's HDI by more than `overlap_threshold`
    (as a fraction of the narrower interval's width) -- i.e. the weaker
    of the two near-duplicate effects is the one that goes.

    Returns (kept_feats, kept_indices).
    """
    mask = (summary["hdi_3%"] > 0) | (summary["hdi_97%"] < 0)
    sig = summary[mask].copy()
    labels = sig.index.tolist()
    indices = [int(lbl.split("[")[1].split("]")[0]) for lbl in labels]

    if len(sig) == 0:
        return [], []

    # process strongest effect first, so a redundant *weaker* duplicate
    # is the one dropped rather than an arbitrary member of the pair
    order = np.argsort(-sig["mean"].abs().to_numpy())

    kept_local = []
    for i in order:
        lo_i, hi_i = sig["hdi_3%"].iloc[i], sig["hdi_97%"].iloc[i]
        redundant_with = None
        for k in kept_local:
            lo_k, hi_k = sig["hdi_3%"].iloc[k], sig["hdi_97%"].iloc[k]
            if _hdi_overlap(lo_i, hi_i, lo_k, hi_k) > overlap_threshold:
                redundant_with = k
                break
        if redundant_with is None:
            kept_local.append(i)
        elif verbose:
            print(
                f"  dropping {feats[indices[i]]!r} (label {labels[i]}) -- "
                f"HDI near-duplicate of {feats[indices[redundant_with]]!r} "
                f"(label {labels[redundant_with]})"
            )

    kept_local = sorted(kept_local)  # restore original feature order
    kept_indices = [indices[i] for i in kept_local]
    kept_feats = [feats[i] for i in kept_indices]
    return kept_feats, kept_indices


def fit_position_pipeline(
    player_data,
    feats,
    gw,
    select_probs_col="league_select_probs",
    position_name="",
    eps=1e-6,
    random_seed=42,
):
    """
    Full pipeline for one position:
      1. Build X (raw features) and select_probs from player_data.
      2. Standardize X.
      3. Fit Dirichlet regression with a weakly-informative prior.
      4. Prune features whose 94% HDI includes zero.
      5. Re-standardize on the pruned feature set and refit.

    Returns a dict with everything needed to reuse this model later
    (predicting alpha_{j,T}^league on a new week's features).
    """
    print(f"\n{'=' * 60}\nPosition: {position_name}\n{'=' * 60}")

    # --- 1. raw data -----------------------------------------------------
    select_probs = player_data[select_probs_col] + eps
    select_probs = (select_probs / select_probs.sum()).to_numpy(dtype=float)

    X_raw = player_data[feats].to_numpy(dtype=float)
    assert not np.isnan(X_raw).any(), f"NaNs in features for {position_name}"
    assert not np.isinf(X_raw).any(), f"Infs in features for {position_name}"
    assert X_raw.shape[0] == select_probs.shape[0], "row mismatch"

    n_players, n_features = X_raw.shape
    print(f"n_players={n_players}, n_features={n_features}")

    # --- 2. standardize ----------------------------------------------------
    X_mean = X_raw.mean(axis=0)
    X_std = X_raw.std(axis=0)
    X_std[X_std == 0] = 1.0
    X = (X_raw - X_mean) / X_std

    # import numpy as np

    # X = data[feats].to_numpy(dtype='float64')

    # print(X)

    rank = np.linalg.matrix_rank(X)
    n_features = X.shape[1]
    cond_number = np.linalg.cond(X)

    print(f"X shape: {X.shape}")
    print(f"rank: {rank}  (full rank would be {n_features})")
    print(f"condition number: {cond_number:.2e}")
    print("  (as a rough guide: >1e3 is concerning, >1e6 is basically singular)")

    if rank < n_features:
        print(
            f"\nWARNING: X is rank-deficient by {n_features - rank} -- "
            f"there is at least one exact linear dependency among your "
            f"standardized features. This alone can cause exactly the "
            f"single-chain-catastrophically-diverges pattern you're seeing, "
            f"independent of the prior."
        )

    # pairwise check for near-exact duplicates (|corr| very close to 1, not just > 0.8)
    corr = np.corrcoef(X, rowvar=False)
    np.fill_diagonal(corr, 0)
    near_dupe = np.argwhere(np.abs(corr) > 0.999)
    if len(near_dupe):
        print("\nNear-exact duplicate columns (|corr| > 0.999):")
        seen = set()
        for i, j in near_dupe:
            if i < j and (i, j) not in seen:
                seen.add((i, j))
                print(f"  columns {i} and {j}: r={corr[i, j]:.6f}")

    # --- 3. initial fit ------------------------------------------------------
    print("Fitting initial model...")
    trace, summary = fit_dirichlet_regression(
        X, select_probs, n_features, random_seed=random_seed
    )

    # az.plot_trace(trace, combined=True)

    # fig = plt.gcf()
    # fig.set_size_inches(14, 3)

    # plt.suptitle(f"Innitial trace Plot of β Coefficients for {position_name} in gw {gw}", y=1.02)
    # plt.tight_layout()
    # plt.show()
    # --- 4. prune ------------------------------------------------------------
    feats_kept, idx_kept = prune_by_hdi(summary, feats)
    print(f"Kept {len(feats_kept)} / {len(feats)} features: {feats_kept}")

    if len(feats_kept) == 0:
        print("WARNING: no features survived pruning -- keeping full set instead")
        feats_kept, idx_kept = feats, list(range(n_features))

    # --- 5. refit on pruned features ------------------------------------------
    X_raw_kept = X_raw[:, idx_kept]
    X_mean_kept = X_raw_kept.mean(axis=0)
    X_std_kept = X_raw_kept.std(axis=0)
    X_std_kept[X_std_kept == 0] = 1.0
    X_kept = (X_raw_kept - X_mean_kept) / X_std_kept

    print("Refitting on pruned feature set...")
    trace_final, summary_final = fit_dirichlet_regression(
        X_kept, select_probs, len(feats_kept), random_seed=random_seed
    )

    # # axes = az.plot_trace(trace_final, combined=True)
    # az.plot_trace(trace_final, combined=True)
    # fig = plt.gcf()
    # fig.set_size_inches(14, 3)

    # plt.suptitle(f"Final trace Plot of β Coefficients for {position_name} in gw {gw}", y=1.02)
    # plt.tight_layout()
    # plt.show()

    return {
        "feats": feats_kept,
        "X_mean": X_mean_kept,
        "X_std": X_std_kept,
        "trace": trace,
        "summary": summary,
        "trace_final": trace_final,
        "summary_final": summary_final,
        "beta_mean": trace_final.posterior["beta"].mean(dim=("chain", "draw")).values,
    }


def alpha_league(fit_result, X_new_raw):
    """
    Apply a fitted position model (dict returned by fit_position_pipeline) to
    new-week raw features, standardizing with the model's *training*
    mean/std, per eq:alpha_league.
    """
    X_std_new = (X_new_raw - fit_result["X_mean"]) / fit_result["X_std"]
    linear = np.clip(X_std_new @ fit_result["beta_mean"], -30, 30)
    return np.exp(linear)


# select_features_by_correlation(X_gk, GK_FEATS)


In [243]:
# for gw in range(19, 39):
#     print(f"\n{'=' * 60}\n Model for round {gw} \n{'=' * 60}")
data_ = X_gk[X_gk["round"] < 6].fillna(0)
# feats = GK_FEATS.copy()
# feats.append("select_prob")
# print(feats)
# feats.remove('xP')
selected_feats = select_features_by_correlation(
    data_, GK_FEATS, redundancy_threshold=0.4, min_corr=0.1, verbose=True
)

X_gk_selected = X_gk[selected_feats]
# league_model = fit_position_pipeline(
#         data, selected_feats, gw, position_name='DEF', random_seed=42
#     )

# print(f"Features for DEF kept in gw {gw} -> {league_model['feats']}")

# league_models[gw] = league_model

  skipping 'round' -- undefined correlation with select_prob (likely constant column)
  skipping 'elo_h' -- undefined correlation with select_prob (likely constant column)
  skipping 'elo_a' -- undefined correlation with select_prob (likely constant column)
  skipping 'team_h_difficulty' -- undefined correlation with select_prob (likely constant column)
  skipping 'team_a_difficulty' -- undefined correlation with select_prob (likely constant column)
  skipping 'win_prob' -- undefined correlation with select_prob (likely constant column)
  skipping 'value' -- undefined correlation with select_prob (likely constant column)
  skipping 'selected_by_percent' -- undefined correlation with select_prob (likely constant column)
  skipping 'transfers_in' -- undefined correlation with select_prob (likely constant column)
  skipping 'transfers_out' -- undefined correlation with select_prob (likely constant column)
  skipping 'xP_rolling_3' -- undefined correlation with select_prob (likely constant

In [ ]:
def fit_dirichlet_regression(X, P_observed):
    """
    Fits a Bayesian Dirichlet regression model using PyMC to estimate beta coefficients.


    Args:
        X (np.ndarray): The feature matrix for a specific position (e.g., GKs).
                        Shape should be (num_players, num_features).
        P_observed (np.ndarray): The observed historical ownership proportions for that position.
                                 Shape should be (num_players,). Must sum to 1.

    Returns:
        pm.backends.base.MultiTrace: The results of the MCMC simulation, containing posterior samples for beta.
    """
    n_players, n_features = X.shape

    with pm.Model() as dirichlet_model:
        # 1. Define priors for our beta coefficients. We'll use a flat (uninformative) prior as mentioned.
        # [span_0](start_span)This corresponds to p(beta) in equation (6)[span_0](end_span).
        # betas = pm.Uniform("betas", lower=-5, upper=5, shape=(n_features,)) #
        betas = pm.Normal("betas", mu=0, sigma=1, shape=(n_features,))

        # 2. Calculate alpha based on the linear model: alpha = exp(X * beta)
        # [span_1](start_span)This is equation (4) in the paper[span_1](end_span).
        alpha = pm.math.exp(pm.math.dot(X, betas)) + 1e-6

        # 3. Define the likelihood of our observed data P_observed
        # The observed ownership proportions are assumed to follow a Dirichlet distribution
        # parameterized by alpha. [span_2](start_span)[span_3](start_span)This is the core of the regression[span_2](end_span)[span_3](end_span).
        pm.Dirichlet("p_obs", a=alpha, observed=P_observed)

        # 4. Run the MCMC sampler to get the posterior distribution of beta
        print("Fitting Dirichlet regression model with PyMC...")
        trace = pm.sample(3000, tune=1000, chains=4, cores=4)
        print("✅ Model fitting complete.")

        # approx = pm.fit(n=2000, method="advi", progressbar=True)
        # trace = approx.sample(2000)

    return trace

In [ ]:
# Fit the model to get beta estimates
trace_gk = fit_dirichlet_regression(X_gk_selected, gk_observed_p)

# Extract the mean of the beta posterior samples to use in Algorithm 1
beta_estimates_gk = trace_gk.posterior["betas"].mean(axis=(0, 1)).values

# trace_gk
az.plot_trace(trace_gk, combined=True)

# Summary with R-hat, ESS, mean, sd, etc.
summary = az.summary(trace_gk, var_names=["betas"], round_to=2)
summary

Fitting Dirichlet regression model with PyMC...


KeyboardInterrupt: 